# Notebook A — Kernels classiques : fondements mathématiques
## Guide complet pour comprendre QMKL depuis zéro

Ce notebook pose **toutes les bases** mathématiques des kernels avant d'aborder le quantique.
Chaque formule est dérivée, chaque concept est illustré avec du code fonctionnel.

**Plan :**
1. Pourquoi les kernels ? La limite du linéaire
2. L'astuce du kernel (Kernel Trick) — théorème de Mercer
3. RKHS — l'espace fonctionnel associé à un kernel
4. SVM dual — dérivation complète depuis le Lagrangien
5. Matrice Gram — propriétés et visualisation
6. Alignment kernel-target — mesurer la qualité d'un kernel


## Section 1 — Pourquoi les kernels ?

### Le problème fondamental : la non-séparabilité linéaire

Un classifieur linéaire cherche un hyperplan $\mathbf{w}^\top \mathbf{x} + b = 0$ qui sépare les classes.
Cela ne fonctionne **que si les données sont linéairement séparables** — ce qui est rarement le cas en pratique.

**Idée clé :** projeter les données dans un espace de dimension supérieure où elles *deviennent* séparables.

Formellement, on définit une **feature map** :
$$\phi : \mathbb{R}^d \longrightarrow \mathcal{H}$$
où $\mathcal{H}$ est un espace de Hilbert (potentiellement de dimension infinie).

**Exemple simple en 1D :**
- Données originales : $x \in \mathbb{R}$, classes $y = +1$ si $x \in [-1, -0.5] \cup [0.5, 1]$, $y = -1$ sinon.
- Non séparable en 1D par un seuil unique.
- Feature map : $\phi(x) = (x, x^2)$ — projeté en 2D, les données deviennent séparables !

**Problème :** si $\mathcal{H}$ est de dimension $D$ très grande (ou infinie), calculer $\phi(x)$ est coûteux ou impossible.
C'est là qu'intervient l'**astuce du kernel**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

np.random.seed(42)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Dataset 1D non-séparable ──────────────────────────────────────────────
x1 = np.concatenate([np.random.uniform(-1, -0.5, 20), np.random.uniform(0.5, 1, 20)])
x2 = np.random.uniform(-0.4, 0.4, 30)
y1, y2 = np.ones(40), -np.ones(30)
X = np.concatenate([x1, x2])
y = np.concatenate([y1, y2])

ax = axes[0]
ax.scatter(X[y==1], np.zeros_like(X[y==1]), c='#e74c3c', s=50, label='Classe +1', zorder=3)
ax.scatter(X[y==-1], np.zeros_like(X[y==-1]), c='#3498db', s=50, marker='s', label='Classe -1', zorder=3)
ax.set_title('Espace original R¹\n→ non-séparable', fontweight='bold')
ax.set_yticks([])
ax.legend(fontsize=8)
ax.text(0, 0.3, 'Aucun seuil ne sépare
les deux classes', ha='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#fff9c4', edgecolor='orange'))

# ── Feature map φ(x) = (x, x²) ──────────────────────────────────────────
ax = axes[1]
phi_pos = np.stack([x1, x1**2], axis=1)
phi_neg = np.stack([x2, x2**2], axis=1)
ax.scatter(phi_pos[:,0], phi_pos[:,1], c='#e74c3c', s=50, label='Classe +1', zorder=3)
ax.scatter(phi_neg[:,0], phi_neg[:,1], c='#3498db', s=50, marker='s', label='Classe -1', zorder=3)
# Hyperplan séparateur dans l'espace projeté
xx = np.linspace(-1, 1, 100)
ax.plot(xx, 0.25 * np.ones_like(xx), 'k--', lw=2, label='Hyperplan séparateur')
ax.set_xlabel('x'), ax.set_ylabel('x²')
ax.set_title('Espace projeté R²\nφ(x) = (x, x²)\n→ séparable !', fontweight='bold')
ax.legend(fontsize=8)

# ── Feature map φ(x) = (x, x², x³) (cercle 2D classique XOR) ───────────
n = 100
angles = np.linspace(0, 2*np.pi, n)
r1 = np.random.uniform(0, 0.5, 50)
r2 = np.random.uniform(0.7, 1.2, 50)
X_xor = np.vstack([np.column_stack([r1*np.cos(np.random.uniform(0,2*np.pi,50)),
                                     r1*np.sin(np.random.uniform(0,2*np.pi,50))]),
                   np.column_stack([r2*np.cos(np.random.uniform(0,2*np.pi,50)),
                                     r2*np.sin(np.random.uniform(0,2*np.pi,50))])])
y_xor = np.array([1]*50 + [-1]*50)
ax = axes[2]
ax.scatter(X_xor[y_xor==1,0], X_xor[y_xor==1,1], c='#e74c3c', s=30, label='Classe +1')
ax.scatter(X_xor[y_xor==-1,0], X_xor[y_xor==-1,1], c='#3498db', s=30, marker='s', label='Classe -1')
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(0.6*np.cos(theta), 0.6*np.sin(theta), 'k--', lw=2, label='Frontière circulaire')
ax.set_title('Dataset "cercles"\n→ non-séparable linéairement', fontweight='bold')
ax.legend(fontsize=8)
ax.set_aspect('equal')

plt.suptitle('Motivation des kernels : la non-séparabilité linéaire', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/A_nonseparable.png', dpi=100, bbox_inches='tight')
plt.show()
print("Observation : dans l'espace original, les données ne sont pas séparables.")
print("Avec φ(x)=(x,x²), on peut tracer un hyperplan séparateur horizontal.")


## Section 2 — L'astuce du kernel (Kernel Trick)

### Le théorème de Mercer

L'idée centrale : on n'a **jamais besoin de calculer $\phi(x)$ explicitement**.

**Définition :** Une fonction $K : \mathbb{R}^d \times \mathbb{R}^d \to \mathbb{R}$ est un **kernel valide** (noyau de Mercer) si et seulement si :
1. **Symétrie :** $K(x, x') = K(x', x)$
2. **Semi-définie positive :** pour tout ensemble fini $\{x_1, ..., x_n\}$, la matrice $K_{ij} = K(x_i, x_j)$ est semi-définie positive ($\mathbf{v}^\top K \mathbf{v} \geq 0$ pour tout $\mathbf{v}$).

**Théorème de Mercer :** Tout kernel valide $K$ admet une décomposition :
$$K(x, x') = \sum_{i=1}^{\infty} \lambda_i \phi_i(x) \phi_i(x') = \langle \phi(x), \phi(x') \rangle_{\mathcal{H}}$$
où $\lambda_i \geq 0$ et $\phi_i$ sont les valeurs/vecteurs propres de l'opérateur intégral associé à $K$.

**Astuce du kernel :** Partout où un algorithme linéaire n'utilise les données que via des produits scalaires $\langle x_i, x_j \rangle$, on peut substituer $K(x_i, x_j)$ pour travailler implicitement dans $\mathcal{H}$.

### Kernels classiques

| Kernel | Formule | Paramètres | Espace implicite |
|--------|---------|------------|-----------------|
| **Linéaire** | $K(x,x') = x^\top x'$ | — | $\mathbb{R}^d$ |
| **Polynomial** | $K(x,x') = (x^\top x' + c)^p$ | $c \geq 0$, $p \in \mathbb{N}$ | Polynômes de degré $\leq p$ |
| **RBF / Gaussien** | $K(x,x') = \exp(-\gamma \|x-x'\|^2)$ | $\gamma > 0$ | Dimension **infinie** (fonctions gaussiennes) |
| **Cosinus** | $K(x,x') = \cos(\omega(x-x'))$ | $\omega$ | Espace de Fourier |

**Intuition du RBF :** $K_{\text{RBF}}(x,x') = 1$ si $x = x'$, décroît exponentiellement avec la distance. C'est une mesure de similarité locale. En développant l'exponentielle en série :
$$e^{-\gamma\|x-x'\|^2} = \sum_{n=0}^{\infty} \frac{(-\gamma)^n \|x-x'\|^{2n}}{n!}$$
ce qui montre que le RBF kernel "voit" tous les polynômes de tous les degrés simultanément.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Définition des 3 kernels ──────────────────────────────────────────────
def k_rbf(x1, x2, gamma=1.0):
    return np.exp(-gamma * (x1 - x2)**2)

def k_poly(x1, x2, c=1.0, p=3):
    return (x1 * x2 + c)**p

def k_cosine(x1, x2, omega=2.0):
    return np.cos(omega * (x1 - x2))

# ── Visualisation sur grille 1D ───────────────────────────────────────────
x_ref = 0.5   # point de référence
x_grid = np.linspace(-3, 3, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

kernels = [
    (k_rbf,     {'gamma': 1.0}, '#e74c3c', 'RBF (γ=1)', 'Localité: décroît avec la distance'),
    (k_poly,    {'c': 1.0, 'p': 3}, '#3498db', 'Polynomial (p=3)', 'Interactions polynomiales x³'),
    (k_cosine,  {'omega': 2.0}, '#2ecc71', 'Cosinus (ω=2)', 'Périodique: structures répétitives'),
]

for ax, (kfn, kargs, color, title, desc) in zip(axes, kernels):
    vals = [kfn(x_ref, x, **kargs) for x in x_grid]
    ax.plot(x_grid, vals, color=color, lw=2.5)
    ax.axvline(x_ref, color='black', ls='--', lw=1.5, label=f'Référence x={x_ref}')
    ax.axhline(0, color='grey', lw=0.5)
    ax.fill_between(x_grid, 0, vals, alpha=0.15, color=color)
    ax.set_title(f'{title}\n{desc}', fontweight='bold', fontsize=10)
    ax.set_xlabel('x''), ax.set_ylabel(f'K({x_ref}, x')')
    ax.legend(fontsize=8)

plt.suptitle('K(x_ref=0.5, x') pour 3 kernels — mesure de similarité', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Matrices kernel 5×5 ───────────────────────────────────────────────────
X5 = np.array([-2., -1., 0., 1., 2.])
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (kfn, kargs, color, title, _) in zip(axes, kernels):
    K = np.array([[kfn(xi, xj, **kargs) for xj in X5] for xi in X5])
    im = ax.imshow(K, cmap='Blues', vmin=0, vmax=1 if 'cos' not in title.lower() else None)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{K[i,j]:.2f}', ha='center', va='center',
                    fontsize=9, color='white' if K[i,j] > 0.6 else 'black')
    ax.set_xticks(range(5)), ax.set_yticks(range(5))
    ax.set_xticklabels([f'{x:.0f}' for x in X5])
    ax.set_yticklabels([f'{x:.0f}' for x in X5])
    ax.set_title(f'Matrice Gram 5×5\n{title}', fontweight='bold')

    # Vérifier PSD: toutes les valeurs propres ≥ 0
    eigenvals = np.linalg.eigvalsh(K)
    print(f'{title}: eigenvalues min={eigenvals.min():.4f} → PSD={eigenvals.min()>=-1e-10}')

plt.tight_layout()
plt.show()


## Section 3 — Reproducing Kernel Hilbert Space (RKHS)

### Définition formelle

À chaque kernel de Mercer $K$ est associé un **espace de Hilbert de fonctions** $\mathcal{H}_K$ appelé RKHS.

**Propriété de reproduction (qui donne le nom) :**
$$f(x) = \langle f, K(\cdot, x) \rangle_{\mathcal{H}_K} \quad \forall f \in \mathcal{H}_K$$

Autrement dit : évaluer une fonction $f$ en un point $x$ revient à prendre le produit scalaire avec la fonction "centrée en $x$" : $K(\cdot, x)$.

**Construction :** $\mathcal{H}_K$ est le complété de l'espace :
$$\left\{ f = \sum_{i=1}^n \alpha_i K(\cdot, x_i) \; \middle| \; n \in \mathbb{N}, \alpha_i \in \mathbb{R}, x_i \in \mathcal{X} \right\}$$
avec le produit scalaire :
$$\left\langle \sum_i \alpha_i K(\cdot, x_i),\; \sum_j \beta_j K(\cdot, x_j) \right\rangle_{\mathcal{H}_K} = \sum_{i,j} \alpha_i \beta_j K(x_i, x_j)$$

### Norme RKHS comme régularisateur

La norme $\|f\|_{\mathcal{H}_K}^2$ mesure la "complexité" de la fonction $f$.

**Exemple pour le RBF :** $\|f\|_{\mathcal{H}_{\text{RBF}}}^2 = \int |\hat{f}(\omega)|^2 e^{\omega^2/(4\gamma)} d\omega$
— cette norme pénalise les oscillations à haute fréquence (fonction lisse = norme faible).

### Théorème du représentant (Representer theorem)

**Théorème :** Pour tout problème d'optimisation de la forme :
$$\min_{f \in \mathcal{H}_K} \sum_{i=1}^n L(y_i, f(x_i)) + \lambda \|f\|_{\mathcal{H}_K}^2$$
la solution optimale est de la forme :
$$f^*(x) = \sum_{i=1}^n \alpha_i K(x, x_i)$$

**Conséquence cruciale :** On ne cherche pas dans un espace infini-dimensionnel — la solution est toujours dans le span fini des $n$ évaluations du kernel. C'est ce qui rend les SVM avec kernel **computationnellement faisables**.

### Lien avec la régularisation de Tikhonov

Le SVM soft-margin résout exactement :
$$\min_{f \in \mathcal{H}_K} \frac{1}{n} \sum_i \max(0, 1 - y_i f(x_i)) + \lambda \|f\|_{\mathcal{H}_K}^2$$
Le paramètre $C = 1/(2\lambda)$ contrôle l'équilibre biais-variance via la norme RKHS.


## Section 4 — SVM avec kernel : dérivation complète depuis le Lagrangien

### Problème primal (hard-margin)

On cherche l'hyperplan de marge maximale dans $\mathcal{H}$ :
$$\min_{\mathbf{w} \in \mathcal{H}, b} \frac{1}{2} \|\mathbf{w}\|^2 \quad \text{s.t.} \quad y_i (\langle \mathbf{w}, \phi(x_i) \rangle + b) \geq 1 \quad \forall i$$

### Lagrangien

$$\mathcal{L}(\mathbf{w}, b, \boldsymbol{\alpha}) = \frac{1}{2}\|\mathbf{w}\|^2 - \sum_{i=1}^n \alpha_i [y_i (\langle\mathbf{w}, \phi(x_i)\rangle + b) - 1]$$

avec multiplicateurs de Lagrange $\alpha_i \geq 0$.

### Conditions KKT (stationnarité)

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = 0 \implies \mathbf{w}^* = \sum_{i=1}^n \alpha_i y_i \phi(x_i)$$
$$\frac{\partial \mathcal{L}}{\partial b} = 0 \implies \sum_{i=1}^n \alpha_i y_i = 0$$

### Problème dual (substitution dans le Lagrangien)

En substituant $\mathbf{w}^* = \sum_i \alpha_i y_i \phi(x_i)$ :

$$\|\mathbf{w}\|^2 = \sum_{i,j} \alpha_i \alpha_j y_i y_j \langle \phi(x_i), \phi(x_j) \rangle = \sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j)$$

Le problème dual devient :
$$\max_{\boldsymbol{\alpha} \geq 0} \sum_{i=1}^n \alpha_i - \frac{1}{2} \sum_{i,j=1}^n \alpha_i \alpha_j y_i y_j K(x_i, x_j) \quad \text{s.t.} \quad \sum_{i=1}^n \alpha_i y_i = 0$$

**Le kernel $K$ apparaît naturellement** — pas besoin de calculer $\phi$ !

### Prédiction

$$f(x) = \text{sign}\left( \sum_{i=1}^n \alpha_i^* y_i K(x_i, x) + b^* \right)$$

où $b^* = y_j - \sum_i \alpha_i y_i K(x_i, x_j)$ pour tout vecteur support $j$ ($\alpha_j > 0$).

### Soft-margin (C-SVM)

On ajoute des variables de marge $\xi_i \geq 0$ ("slack variables") :
$$\min_{\mathbf{w},b,\boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_i \xi_i \quad \text{s.t.} \quad y_i(\langle\mathbf{w},\phi(x_i)\rangle+b) \geq 1 - \xi_i, \; \xi_i \geq 0$$

Dans le dual : la seule différence est $0 \leq \alpha_i \leq C$. **$C$ borne les $\alpha_i$ par le haut.**


In [ ]:
import numpy as np
from scipy.optimize import minimize
from sklearn.svm import SVC
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt

np.random.seed(42)

# ── Dataset "cercles" non-séparable linéairement ──────────────────────────
X, y = make_circles(n_samples=60, noise=0.1, factor=0.4, random_state=42)
y = 2*y - 1   # convertir {0,1} → {-1,+1}

# ── Implémentation du problème dual SVM depuis zéro ──────────────────────
def rbf_kernel(X1, X2, gamma=1.0):
    '''Calcule la matrice kernel RBF entre X1 et X2.'''
    # ||x_i - x_j||² = ||x_i||² + ||x_j||² - 2 x_i·x_j
    sq1 = np.sum(X1**2, axis=1, keepdims=True)   # (n1, 1)
    sq2 = np.sum(X2**2, axis=1, keepdims=True).T  # (1, n2)
    cross = X1 @ X2.T                              # (n1, n2)
    return np.exp(-gamma * (sq1 + sq2 - 2*cross))

n = len(X)
K = rbf_kernel(X, X, gamma=2.0)
C = 1.0

# Matrice Q_ij = y_i * y_j * K(x_i, x_j)
Q = (y.reshape(-1,1) * y.reshape(1,-1)) * K   # (n, n)

# Objectif dual : max Σα_i - (1/2) α^T Q α = min (1/2) α^T Q α - Σα_i
def objective(alpha):
    return 0.5 * alpha @ Q @ alpha - np.sum(alpha)

def gradient(alpha):
    return Q @ alpha - np.ones(n)

# Contraintes: Σ α_i y_i = 0 et 0 ≤ α_i ≤ C
constraints = {'type': 'eq', 'fun': lambda a: np.dot(a, y), 'jac': lambda a: y}
bounds = [(0, C)] * n

alpha0 = np.zeros(n)
result = minimize(objective, alpha0, jac=gradient,
                  method='SLSQP', bounds=bounds, constraints=constraints,
                  options={'ftol': 1e-8, 'maxiter': 1000})

alphas = result.x
# Vecteurs support : α_i > seuil numérique
sv_mask = alphas > 1e-4
print(f"Vecteurs support trouvés: {sv_mask.sum()} / {n}")

# Calculer b* : moyenne sur les vecteurs support
b_vals = [y[j] - np.sum(alphas * y * K[:, j]) for j in np.where(sv_mask)[0]]
b_star = np.mean(b_vals)

# ── Prédiction sur grille ─────────────────────────────────────────────────
xx, yy = np.meshgrid(np.linspace(-1.5,1.5,100), np.linspace(-1.5,1.5,100))
X_grid = np.column_stack([xx.ravel(), yy.ravel()])
K_grid = rbf_kernel(X_grid, X, gamma=2.0)  # (10000, n)
f_grid = (K_grid @ (alphas * y)) + b_star
Z = f_grid.reshape(xx.shape)

# ── Comparaison avec sklearn ──────────────────────────────────────────────
svm_sklearn = SVC(kernel='rbf', C=1.0, gamma=2.0)
svm_sklearn.fit(X, y)
Z_sklearn = svm_sklearn.decision_function(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, Z_plot, title in zip(axes,
        [Z, Z_sklearn],
        ['SVM dual implémenté depuis zéro\n(scipy.optimize.minimize)',
         'SVM sklearn (référence)']):
    ax.contourf(xx, yy, Z_plot, levels=[-1e9, 0, 1e9], colors=['#aed6f1','#f1948a'], alpha=0.4)
    ax.contour(xx, yy, Z_plot, levels=[0], colors='black', linewidths=2)
    ax.contour(xx, yy, Z_plot, levels=[-1, 1], colors='black', linewidths=1, linestyles='--')
    ax.scatter(X[y==1,0], X[y==1,1], c='#e74c3c', s=60, zorder=3, label='+1')
    ax.scatter(X[y==-1,0], X[y==-1,1], c='#3498db', s=60, marker='s', zorder=3, label='-1')
    if 'sklearn' not in title:
        ax.scatter(X[sv_mask,0], X[sv_mask,1], s=200, facecolors='none',
                   edgecolors='gold', linewidths=2.5, zorder=4, label='Vecteurs support')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Vérifier l'accord entre les deux
acc_own = np.mean(np.sign(Z.ravel()[::10]) == y.repeat(10)[:len(Z.ravel()[::10])])
pred_own = np.sign((rbf_kernel(X, X, gamma=2.0) @ (alphas * y)) + b_star)
pred_sklearn = svm_sklearn.predict(X)
print(f"Accord prédictions (train): {np.mean(pred_own == pred_sklearn)*100:.1f}%")


## Section 5 — Matrice Gram : propriétés et visualisation

### Définition

Pour un ensemble de $n$ points $\{x_1, ..., x_n\}$ et un kernel $K$, la **matrice Gram** (ou matrice kernel) est :
$$\mathbf{K} \in \mathbb{R}^{n \times n}, \quad K_{ij} = K(x_i, x_j)$$

### Propriétés fondamentales

1. **Symétrie :** $\mathbf{K} = \mathbf{K}^\top$ (car $K(x,x') = K(x',x)$)
2. **Semi-définie positive (PSD) :** $\forall \mathbf{v} \in \mathbb{R}^n : \mathbf{v}^\top \mathbf{K} \mathbf{v} \geq 0$
   - Preuve : $\mathbf{v}^\top \mathbf{K} \mathbf{v} = \sum_{i,j} v_i v_j K(x_i,x_j) = \|\sum_i v_i \phi(x_i)\|^2_{\mathcal{H}} \geq 0$
3. **Diagonale = norme au carré :** $K_{ii} = \|\phi(x_i)\|_{\mathcal{H}}^2$ (pour RBF: $K_{ii} = 1$ toujours)
4. **Inégalité de Cauchy-Schwarz :** $|K(x,x')|^2 \leq K(x,x) \cdot K(x',x')$

### Centrage de la matrice kernel

On veut que les features $\phi(x_i)$ soient de moyenne nulle dans $\mathcal{H}$.
Le kernel centré est :
$$\tilde{K}_{ij} = K_{ij} - \frac{1}{n}\sum_k K_{ik} - \frac{1}{n}\sum_k K_{kj} + \frac{1}{n^2}\sum_{k,l} K_{kl}$$

En notation matricielle :
$$\tilde{\mathbf{K}} = \mathbf{H} \mathbf{K} \mathbf{H}, \quad \text{où} \quad \mathbf{H} = \mathbf{I} - \frac{1}{n}\mathbf{1}\mathbf{1}^\top$$

$\mathbf{H}$ est la matrice de centrage. **Centrer est essentiel** pour que l'alignment kernel-target soit une mesure pure de corrélation (section suivante).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Charger et préparer les données ──────────────────────────────────────
data = load_breast_cancer()
X_raw, y_raw = data.data, data.target
y_raw = 2*y_raw - 1   # {0,1} → {-1,+1}

# Standardisation + PCA à 4 composantes (simule Q=4 qubits)
scaler = StandardScaler()
pca = PCA(n_components=4)
X_std = scaler.fit_transform(X_raw)
X4 = pca.fit_transform(X_std)
# Normaliser dans [0, 2] (convention circuits quantiques)
from sklearn.preprocessing import MinMaxScaler
X4 = MinMaxScaler(feature_range=(0, 2)).fit_transform(X4)

# Prendre un sous-ensemble de 50 points pour visualiser
idx = np.random.RandomState(42).choice(len(X4), 50, replace=False)
X50, y50 = X4[idx], y_raw[idx]
n = len(X50)

# ── Calculer les matrices kernel ──────────────────────────────────────────
def K_rbf(X, gamma=1.0):
    sq = np.sum(X**2, axis=1, keepdims=True)
    dist2 = sq + sq.T - 2*(X@X.T)
    return np.exp(-gamma * dist2)

def K_ZZ(X, alpha=1.0):
    '''ZZ kernel analytique: K = ∏_k cos²(α(x_ik - x_jk)) × ∏_{k<l} cos²(α(x_ik*x_il - x_jk*x_jl))'''
    n, d = X.shape
    K = np.ones((n, n))
    for k in range(d):
        diff = X[:, k:k+1] - X[:, k].reshape(1, -1)
        K *= np.cos(alpha * diff)**2
    for k in range(d):
        for l in range(k+1, d):
            cross_i = X[:, k] * X[:, l]
            diff_cross = cross_i.reshape(-1,1) - cross_i.reshape(1,-1)
            K *= np.cos(alpha * diff_cross)**2
    return K

def center_kernel(K):
    '''Centrage: K_c = H K H, H = I - (1/n) 11^T'''
    n = len(K)
    H = np.eye(n) - np.ones((n,n)) / n
    return H @ K @ H

K1 = K_rbf(X50, gamma=0.5)
K2 = K_ZZ(X50, alpha=1.0)
K1_c = center_kernel(K1)
K2_c = center_kernel(K2)

# ── Visualisation : 4 matrices ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Trier les points par label pour mieux voir la structure de bloc
sort_idx = np.argsort(y50)
titles = ['K_RBF (brut)', 'K_ZZ (brut)', 'K_RBF (centré)', 'K_ZZ (centré)']
mats = [K1, K2, K1_c, K2_c]

for ax, mat, title in zip(axes.ravel(), mats, titles):
    M = mat[np.ix_(sort_idx, sort_idx)]  # réordonner par classe
    im = ax.imshow(M, cmap='RdBu_r', aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046)
    # Ligne de séparation des classes
    n_pos = np.sum(y50[sort_idx] == 1)
    ax.axhline(n_pos-0.5, color='gold', lw=2)
    ax.axvline(n_pos-0.5, color='gold', lw=2)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.text(n_pos/2, -3, 'Classe +1', ha='center', fontsize=8, color='#c0392b')
    ax.text(n_pos + (n-n_pos)/2, -3, 'Classe -1', ha='center', fontsize=8, color='#2980b9')

    # Vérifier PSD
    eigs = np.linalg.eigvalsh(mat)
    print(f'{title}: min eigenvalue = {eigs.min():.6f} (PSD: {eigs.min() > -1e-9})')

plt.suptitle('Matrices kernel sur 50 patients (Breast Cancer, Q=4)
'
             'Ligne dorée = séparation des classes', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nObservation : après centrage, la structure de bloc (classes) est plus apparente.")
print("La matrice K_ZZ centrée montre un bon alignement avec les labels.")


## Section 6 — Alignment kernel-target : mesurer la qualité d'un kernel

### Définition du Kernel-Target Alignment (KTA)

L'**alignment** mesure la similitude entre la matrice kernel $K$ et la matrice idéale $\mathbf{y}\mathbf{y}^\top$ (où $y_i \in \{-1, +1\}$).

$$A(K, \mathbf{y}\mathbf{y}^\top) = \frac{\langle K, \mathbf{y}\mathbf{y}^\top \rangle_F}{\|K\|_F \cdot \|\mathbf{y}\mathbf{y}^\top\|_F}$$

Le **produit de Frobenius** est : $\langle A, B \rangle_F = \sum_{i,j} A_{ij} B_{ij} = \text{trace}(A^\top B)$

**Intuition :** $\mathbf{y}\mathbf{y}^\top$ est la matrice kernel "parfaite" — elle vaut $+1$ si deux points sont dans la même classe, $-1$ sinon. Un kernel avec fort alignment sépare bien les classes.

### Version centrée (Cortes et al., 2012)

On utilise les **kernels centrés** pour éliminer le biais constant :
$$\hat{A}(K, \mathbf{y}\mathbf{y}^\top) = \frac{\langle \tilde{K}, \tilde{\mathbf{y}\mathbf{y}^\top} \rangle_F}{\sqrt{\langle \tilde{K}, \tilde{K} \rangle_F \cdot \langle \tilde{\mathbf{y}\mathbf{y}^\top}, \tilde{\mathbf{y}\mathbf{y}^\top} \rangle_F}}$$

### Lien avec AUC

**Théorème (Shawe-Taylor & Kandola, 2002) :** L'alignment est une borne supérieure sur la marge du SVM optimal, qui est elle-même liée à l'AUC. En pratique : **alignment élevé ≈ AUC élevée** pour les SVM à kernel.

### Utilisation dans QMKL

L'alignment est le fondement de la stratégie **Centered Alignment** (notebook C) :
$$w^* = \arg\max_{w \geq 0, \sum w_m = 1} \hat{A}\left(\sum_m w_m K_m,\; \mathbf{y}\mathbf{y}^\top\right)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

np.random.seed(42)

# ── Préparer les données ──────────────────────────────────────────────────
data = load_breast_cancer()
X_raw, y_raw = data.data, data.target
y_pm = 2*y_raw - 1   # {0,1} → {-1,+1}

X_std = StandardScaler().fit_transform(X_raw)
X4 = PCA(n_components=4).fit_transform(X_std)
X4 = MinMaxScaler(feature_range=(0,2)).fit_transform(X4)

# Sous-ensemble pour les calculs (200 points)
idx = np.random.choice(len(X4), 200, replace=False)
X, y = X4[idx], y_pm[idx]
n = len(X)

# ── Fonctions d'alignment ─────────────────────────────────────────────────
def center_kernel(K):
    n = len(K)
    H = np.eye(n) - np.ones((n,n))/n
    return H @ K @ H

def frobenius_ip(A, B):
    return np.sum(A * B)

def centered_alignment(K, y):
    '''Alignment centré entre kernel K et cible yy^T'''
    Kc = center_kernel(K)
    Y = np.outer(y, y)    # matrice target n×n
    Yc = center_kernel(Y)
    num = frobenius_ip(Kc, Yc)
    denom = np.sqrt(frobenius_ip(Kc, Kc) * frobenius_ip(Yc, Yc))
    return num / denom if denom > 0 else 0.0

# ── Calculer alignment pour plusieurs kernels ────────────────────────────
def K_rbf(X, gamma):
    sq = np.sum(X**2, axis=1, keepdims=True)
    return np.exp(-gamma * (sq + sq.T - 2*(X@X.T)))

def K_ZZ(X, alpha):
    n, d = X.shape
    K = np.ones((n, n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha * diff)**2
    for k in range(d):
        for l in range(k+1, d):
            cross = X[:,k]*X[:,l]
            diff = cross.reshape(-1,1) - cross.reshape(1,-1)
            K *= np.cos(alpha * diff)**2
    return K

def K_poly(X, p, c=1.0):
    return (X @ X.T / X.shape[1] + c)**p

# Gamme de paramètres
gammas_rbf = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
alphas_ZZ  = [0.5, 1.0, 2.0, 3.0, 4.0]
polys      = [1, 2, 3, 4, 5]

results = []
for g in gammas_rbf:
    K = K_rbf(X, g)
    a = centered_alignment(K, y)
    # Estimer AUC par cross-val (5-fold)
    auc = cross_val_score(SVC(kernel='precomputed', C=1.0), K, y,
                          cv=5, scoring='roc_auc').mean()
    results.append({'label': f'RBF γ={g}', 'align': a, 'auc': auc, 'color': '#e74c3c', 'marker': 'o'})

for a_val in alphas_ZZ:
    K = K_ZZ(X, a_val)
    a = centered_alignment(K, y)
    auc = cross_val_score(SVC(kernel='precomputed', C=1.0), K, y,
                          cv=5, scoring='roc_auc').mean()
    results.append({'label': f'ZZ α={a_val}', 'align': a, 'auc': auc, 'color': '#3498db', 'marker': 's'})

for p in polys:
    K = K_poly(X, p)
    # Forcer PSD
    eigs = np.linalg.eigvalsh(K)
    if eigs.min() < 0:
        K += (-eigs.min() + 1e-8) * np.eye(n)
    a = centered_alignment(K, y)
    auc = cross_val_score(SVC(kernel='precomputed', C=1.0), K, y,
                          cv=5, scoring='roc_auc').mean()
    results.append({'label': f'Poly p={p}', 'align': a, 'auc': auc, 'color': '#2ecc71', 'marker': '^'})

# ── Tracer alignment vs AUC ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for r in results:
    ax.scatter(r['align'], r['auc'], color=r['color'], marker=r['marker'], s=80, zorder=3)
    ax.annotate(r['label'], (r['align'], r['auc']),
                textcoords='offset points', xytext=(5,3), fontsize=7)

# Droite de régression
aligns = [r['align'] for r in results]
aucs   = [r['auc']   for r in results]
coeffs = np.polyfit(aligns, aucs, 1)
xfit = np.linspace(min(aligns), max(aligns), 100)
ax.plot(xfit, np.polyval(coeffs, xfit), 'k--', lw=1.5, label=f'Régression linéaire (R²={np.corrcoef(aligns,aucs)[0,1]**2:.3f})')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0],marker='o',color='w',markerfacecolor='#e74c3c',markersize=10,label='RBF'),
    Line2D([0],[0],marker='s',color='w',markerfacecolor='#3498db',markersize=10,label='ZZ'),
    Line2D([0],[0],marker='^',color='w',markerfacecolor='#2ecc71',markersize=10,label='Polynomial'),
    Line2D([0],[0],color='k',ls='--',label='Régression'),
]
ax.legend(handles=legend_elements, fontsize=9)
ax.set_xlabel('Centered Alignment A(K, yy^T)', fontsize=11)
ax.set_ylabel('AUC (cross-val 5-fold)', fontsize=11)
ax.set_title('Alignment vs AUC — corrélation sur Breast Cancer (N=200, Q=4)', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Corrélation Pearson alignment–AUC: {np.corrcoef(aligns,aucs)[0,1]:.3f}")
print("→ Un alignment élevé prédit bien une AUC élevée.")
print("→ C'est la base de Centered Alignment MKL (Notebook C).")
